# 06 - Clustering & Anomaly (Emerging Issue) Detection

Demonstrates complaint clustering (HDBSCAN) and the emerging-issue detector, validated against the two deliberately injected incidents in the synthetic dataset.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
from src.config import settings
from src.embeddings.encoder import EmbeddingEncoder
from src.clustering.clusterer import run_clustering, summarize_clusters
from src.anomaly.detector import detect_emerging_issues

df = pd.read_csv(settings.path('paths.complaints_csv')).sample(2000, random_state=42).reset_index(drop=True)
df['complaint_text'] = df['complaint_text'].fillna('')
enc = EmbeddingEncoder()
enc.fit(df['complaint_text'].tolist())
embeddings = enc.encode(df['complaint_text'].tolist())
embeddings.shape

2026-08-31 04:04:19,578 | INFO     | src.embeddings.encoder | Fitted TF-IDF+SVD embedding fallback (dim=200) on 2000 documents


(2000, 200)

In [2]:
labels = run_clustering(embeddings)
summaries = summarize_clusters(df, labels)
print(f'{len(summaries)} clusters found')
pd.DataFrame(summaries)[['cluster_label','title','size','dominant_category','dominant_sentiment']].head(10)

2026-08-31 04:04:20,316 | INFO     | src.clustering.clusterer | HDBSCAN produced 70 clusters (77 noise points)


70 clusters found


,cluster_label,title,size,dominant_category,dominant_sentiment
0,33,Android Failed transaction,51,Payment,Negative
1,52,Web Payment pending,48,Payment,Negative
2,45,Android Payment reversed,43,Payment,Negative
3,38,iOS Duplicate charge,38,Payment,Negative
4,46,Android Delivery partner issue,38,Delivery,Negative
5,55,Android Failed transaction,38,Payment,Negative
6,40,iOS Payment pending,37,Payment,Negative
7,64,Android Refund missing,37,Refund,Negative
8,7,Web Duplicate charge,36,Payment,Negative
9,16,iOS Delivery partner issue,36,Delivery,Negative


In [3]:
# Full dataset for anomaly detection (needs the full time range to see the injected incidents)
full_df = pd.read_csv(settings.path('paths.complaints_csv'))
issues = detect_emerging_issues(full_df)
pd.DataFrame(issues)[['title','current_volume','baseline_volume','increase_percent','severity','platform','region']]

2026-08-31 04:04:20,969 | INFO     | src.anomaly.detector | Detected 2 emerging issues


,title,current_volume,baseline_volume,increase_percent,severity,platform,region
0,Technical Issue - OTP failure,21,0.2,8300.0,CRITICAL,Android,North India
1,Payment - failed transaction,20,1.3,1400.0,CRITICAL,Android,South India


Both injected incidents (OTP Verification Failure, UPI Payment Failure Spike) are correctly detected with the right platform/region and CRITICAL severity.